# Download Instructions

This page covers how to download the datasets required by ROMS-Tools.
For an overview of all datasets and their required fields, see the [Datasets](datasets_overview.rst) page.

## Downloading GSHHG coastline data

1. Download the GSHHG shapefile package [gshhg-shp-2.3.7.zip](https://www.ngdc.noaa.gov/mgg/shorelines/data/gshhg/latest/) from the NOAA website.

2. After unzipping, you will find a directory named `GSHHS_shp` containing five subdirectories (`c`, `f`, `h`, `i`, `l`). Each corresponds to a different resolution of the coastline data:

   - **f** (full): highest-resolution, original dataset  
   - **h** (high): ~80% reduction in detail and size  
   - **i** (intermediate): additional ~80% reduction  
   - **l** (low): additional ~80% reduction  
   - **c** (crude): additional ~80% reduction  

3. For `ROMS-Tools`, you only need the **Level-1** (`L1`) shapefiles, which contain coastline polygons. (`L2`–`L6` represent lakes, rivers, and other inland water bodies and are not required.)

   Make sure all four `L1` companion files are present, for example:

   - `GSHHS_f_L1.dbf`  
   - `GSHHS_f_L1.prj`  
   - `GSHHS_f_L1.shp`  
   - `GSHHS_f_L1.shx`  

   Even though you only point ROMS-Tools to the `.shp` file (e.g., `GSHHS_f_L1.shp`), the other files must be in the same directory. See [this notebook](https://roms-tools.readthedocs.io/en/latest/grid.html) for an example.

## Downloading GLORYS data

You can download GLORYS data from the [Copernicus Marine Data Store](https://data.marine.copernicus.eu/products).  
To access the data, [register for a Copernicus Marine Service account](https://help.marine.copernicus.eu/en/articles/4220332-how-to-sign-up-for-copernicus-marine-service) to obtain a username and password.

Once registered, install the `copernicusmarine` package to download the datasets:

```bash
pip install copernicusmarine
```

In [1]:
import copernicusmarine

/Users/noraloose/miniconda3/envs/romstools-test/lib/python3.13/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


When you first log in with `copernicusmarine`, your credentials are saved in a `.copernicusmarine-credentials` file. This one-time setup gives you seamless access to all Copernicus Marine services without re-entering credentials.

```python
copernicusmarine.login(username="YOUR_USERNAME", password="YOUR_PASSWORD")
```

### Downloading global data
This example demonstrates how to download the global GLORYS dataset for a specified time range, defined by `start_time` and `end_time`. In this case, we select January 2012.

In [2]:
from datetime import datetime

In [3]:
start_time = datetime(2012, 1, 1)
end_time = datetime(2012, 2, 1)

In [4]:
%%time

copernicusmarine.subset(
    dataset_id="cmems_mod_glo_phy_my_0.083deg_P1D-m",
    variables=["thetao", "so", "uo", "vo", "zos"],
    minimum_longitude=None, # global data
    maximum_longitude=None, # global data
    minimum_latitude=None, # global data
    maximum_latitude=None, # global data 
    start_datetime=start_time,
    end_datetime=end_time,
    coordinates_selection_method="outside",
    output_filename = "global_GLORYS_Jan2012.nc",
    output_directory = "source-data"
)

INFO - 2025-09-23T20:35:31Z - Selected dataset version: "202311"
INFO - 2025-09-23T20:35:31Z - Selected dataset part: "default"
INFO - 2025-09-23T20:35:33Z - Starting download. Please wait...


  0%|          | 0/6761 [00:00<?, ?it/s]

INFO - 2025-09-24T18:49:02Z - Successfully downloaded to source-data/global_GLORYS_Jan2012.nc


CPU times: user 17min 22s, sys: 15min 31s, total: 32min 53s
Wall time: 22h 13min 35s


ResponseSubset(file_path=PosixPath('source-data/global_GLORYS_Jan2012.nc'), output_directory=PosixPath('source-data'), filename='global_GLORYS_Jan2012.nc', file_size=138819.17330534352, data_transfer_size=630106.1276335877, variables=['thetao', 'so', 'uo', 'vo', 'zos'], coordinates_extent=[GeographicalExtent(minimum=-180.0, maximum=179.9166717529297, unit='degrees_east', coordinate_id='longitude'), GeographicalExtent(minimum=-80.0, maximum=90.0, unit='degrees_north', coordinate_id='latitude'), TimeExtent(minimum='2012-01-01T00:00:00+00:00', maximum='2012-02-02T00:00:00+00:00', unit='iso8601', coordinate_id='time'), GeographicalExtent(minimum=0.49402499198913574, maximum=5727.9169921875, unit='m', coordinate_id='depth')], status='000', message='The request was successful.', file_status='DOWNLOADED')

### Downloading a spatial subset

If you don’t want to download the entire *global* dataset (which can be very time-consuming) you can instead download a **spatial subset** of GLORYS data for a specific domain. This requires specifying `minimum_longitude`, `maximum_longitude`, `minimum_latitude`, and `maximum_latitude`.

Because ROMS grids (at least those created by ROMS-Tools) are **not regular lat-lon grids**, determining these bounds is not straightforward. Additionally, ROMS-Tools requires a **safety margin** to perform [lateral fill](https://roms-tools.readthedocs.io/en/latest/methods.html#multigrid-method-for-filling-land-values) and regridding, which helps prevent boundary artifacts. ROMS-Tools provides a function that can compute appropriate bounds given a grid.

In [5]:
from roms_tools import Grid

In [6]:
grid = Grid(
    nx=100,  # number of grid points in x-direction
    ny=80,  # number of grid points in y-direction
    size_x=2000,  # domain size in x-direction (in km)
    size_y=1600,  # domain size in y-direction (in km)
    center_lon=-89,  # longitude of the center of the domain
    center_lat=24,  # latitude of the center of the domain
    rot=0,  # rotation of the grid (in degrees)
    N=20,  # number of vertical layers
)

In [7]:
from roms_tools import get_glorys_bounds

In [8]:
bounds = get_glorys_bounds(grid)

In [9]:
bounds

{'minimum_latitude': 14.75,
 'maximum_latitude': 33.0,
 'minimum_longitude': 258.75,
 'maximum_longitude': 283.25}

In [10]:
%%time

copernicusmarine.subset(
    dataset_id="cmems_mod_glo_phy_my_0.083deg_P1D-m",
    variables=["thetao", "so", "uo", "vo", "zos"],
    **bounds, # regional data
    start_datetime=start_time,
    end_datetime=end_time,
    coordinates_selection_method="outside",
    output_filename = "GoM_GLORYS_Jan2012.nc",
    output_directory = "source-data"
)

INFO - 2025-09-24T20:23:38Z - Selected dataset version: "202311"
2025-09-24 20:23:38 - INFO - Selected dataset version: "202311"
INFO - 2025-09-24T20:23:38Z - Selected dataset part: "default"
2025-09-24 20:23:38 - INFO - Selected dataset part: "default"
INFO - 2025-09-24T20:23:40Z - Starting download. Please wait...
2025-09-24 20:23:40 - INFO - Starting download. Please wait...


  0%|          | 0/584 [00:00<?, ?it/s]

INFO - 2025-09-24T21:03:21Z - Successfully downloaded to source-data/GoM_GLORYS_Jan2012.nc
2025-09-24 21:03:21 - INFO - Successfully downloaded to source-data/GoM_GLORYS_Jan2012.nc


CPU times: user 1min 21s, sys: 1min 8s, total: 2min 30s
Wall time: 39min 50s


ResponseSubset(file_path=PosixPath('source-data/GoM_GLORYS_Jan2012.nc'), output_directory=PosixPath('source-data'), filename='GoM_GLORYS_Jan2012.nc', file_size=1021.8164351145039, data_transfer_size=52508.84396946565, variables=['thetao', 'so', 'uo', 'vo', 'zos'], coordinates_extent=[GeographicalExtent(minimum=-101.25, maximum=-76.75, unit='degrees_east', coordinate_id='longitude'), GeographicalExtent(minimum=14.75, maximum=33.0, unit='degrees_north', coordinate_id='latitude'), TimeExtent(minimum='2012-01-01T00:00:00+00:00', maximum='2012-02-02T00:00:00+00:00', unit='iso8601', coordinate_id='time'), GeographicalExtent(minimum=0.49402499198913574, maximum=5727.9169921875, unit='m', coordinate_id='depth')], status='000', message='The request was successful.', file_status='DOWNLOADED')

## Downloading the Unified BGC Dataset

This section demonstrates how to download a **unified biogeochemical (BGC) climatology**, which integrates multiple observational and model-based sources:

- **Nutrients (NO₃⁻, PO₄³⁻, SiO₄⁴⁻)** and **dissolved oxygen** from the 2018 **World Ocean Atlas**
- **Dissolved iron (Fe)** and **nitrous oxide (N₂O)** from **in-situ measurements**
- **Dissolved inorganic carbon (DIC)** and **total alkalinity (ALK)** from the **GLODAPv2** global product
- **Other nutrients** (ammonium NH₄⁺, nitrite NO₂⁻, organic nitrogen) and **dissolved organic matter (DOM)** from **CESM model simulations**

The dataset is hosted on **Google Drive** and can be downloaded using the following procedure.

In [11]:
import gdown
import os

In [12]:
url = "https://drive.google.com/uc?id=1wUNwVeJsd6yM7o-5kCx-vM3wGwlnGSiq"

In [13]:
output_dir = "source-data"

In [14]:
os.makedirs(output_dir, exist_ok=True)

In [15]:
gdown.download(url, f"{output_dir}/BGCdataset.nc", quiet=False)

Downloading...
From (original): https://drive.google.com/uc?id=1wUNwVeJsd6yM7o-5kCx-vM3wGwlnGSiq
From (redirected): https://drive.google.com/uc?id=1wUNwVeJsd6yM7o-5kCx-vM3wGwlnGSiq&confirm=t&uuid=7bd915fc-672a-4b7f-b5f1-5282d7f3cb7a
To: /Users/noraloose/roms-tools/docs/source-data/BGCdataset.nc
100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 21.4G/21.4G [32:01<00:00, 11.1MB/s]


'source-data/BGCdataset.nc'

### Handling File Download Limits

<div class="alert alert-info">
Note
    
If you encounter a `FileURLRetrievalError`, it usually means the file has been accessed or downloaded too many times recently. This often happens with large files or files shared by many users.  

**Workaround:** Download the file manually using the following link: [unified BGC dataset](https://drive.google.com/uc?id=1wUNwVeJsd6yM7o-5kCx-vM3wGwlnGSiq)

After downloading, place the file in the appropriate directory for your workflow.

</div>



## Downloading WOA salinity data

This section instructs how to download the World Ocean Atlas salinity data from the [NOAA website](https://www.ncei.noaa.gov/products/world-ocean-atlas). It is a collection of salinity (and other variables) means based on profile data from the World Ocean Database (WOD). The salinity data used in `ROMS-Tools`, needs to be gridded data. The `s_an` variable provided is the 'Objectively analyzed mean fields for sea_water_salinity' and is needed by `ROMS-Tools`.

To download the needed 12 months of data for a climatology record, files with suffixes from `s01-s12` are needed. In [this notebook](https://roms-tools.readthedocs.io/en/latest/surface_forcing.html), we use the decadal averaged data (i.e. files with suffix including `decav`.
- The 2018 Atlas from the notebook mentioned above can be found [here](https://www.ncei.noaa.gov/data/oceans/woa/WOA18/DATA/salinity/netcdf/decav/0.25/).
- Likewise, the 2023 Atlas can be found [here](https://www.ncei.noaa.gov/data/oceans/woa/WOA23/DATA/salinity/netcdf/decav/0.25/). 

## Downloading the MBL CO2 Dataset

This section demonstrates how to download a **time-varying CO2 dataset from NOAA's GML, Marine Boundary Layer Reference**, which integrates observations from a subset of sites from the their Cooperative Global Air Sampling Network. After processing, their data are available approximately weekly (7.6 days). 

The dataset can be downloaded using the following procedure.

In [16]:
import urllib.request

In [17]:
url = "https://gml.noaa.gov/ccgg/mbl/tmp/co2_GHGreference.1785677502_surface.txt"

In [18]:
urllib.request.urlretrieve(url, "co2_GHGreference.1785677502_surface.txt")

('co2_GHGreference.1785677502_surface.txt',
 <http.client.HTTPMessage at 0x1546fb9b1450>)

## Downloading the OceanSODA-ETHZ Dataset

This section demonstrates how to download a **monthly dataset from NOAA's NCEI**. This dataset is calculated from machine learning estimates of Total Alkalinity (TA) and the fugacity of carbon dioxide (fCO2), and the sea surface DIC and ALK data are used for restoring forces in `ROMS`. 

The dataset can be downloaded using the following procedure.

In [19]:
import urllib.request

In [20]:
url = "https://www.ncei.noaa.gov/data/oceans/archive/arc0160/0220059/6.6/data/0-data/OceanSODA_ETHZ-v2025.OCADS.01-1982-2024.nc"

In [21]:
urllib.request.urlretrieve(url, "OceanSODA_ETHZ-v2025.OCADS.01-1982-2024.nc")

('OceanSODA_ETHZ-v2025.OCADS.01-1982-2024.nc',
 <http.client.HTTPMessage at 0x1546fb774c30>)

## Downloading GloFAS data

GloFAS v4.0 provides **daily global river discharge** and can be used as an alternative to Dai & Trenberth when creating river forcing. Raw GloFAS fields must be preprocessed into a coastal station NetCDF before use with `ROMS-Tools`.

### 1. Create an EWDS account

GloFAS lives on the Copernicus Early Warning Data Store (EWDS), not the standard Climate Data Store (CDS).

1. Register for an [ECMWF account](https://www.ecmwf.int/).
2. Open the [GloFAS historical dataset page](https://ewds.climate.copernicus.eu/datasets/cems-glofas-historical) and accept the licence.
3. Follow the [EWDS download instructions](https://confluence.ecmwf.int/display/CEMS/EWDS+-+How+to+Download) and set up API access.

Install the CDS API client:

```bash
pip install cdsapi
```

Configure `~/.cdsapirc` to point at the **EWDS** API endpoint (not the CDS one):

```text
url: https://ewds.climate.copernicus.eu/api
key: <your-ewds-api-key>
```

### 2. Download historical discharge

Download **GloFAS v4.0** consolidated river discharge (`river_discharge_in_the_last_24_hours` / `dis24`). Prefer **one NetCDF file per year**, which matches the layout expected by the preprocessing notebook (`glofas_v4_discharge_YYYY.nc`).

You can use the EWDS web form, or retrieve programmatically:


In [ ]:
import cdsapi

# Example: download one year of GloFAS v4.0 historical discharge.
# Point cdsapi at the EWDS endpoint (see ~/.cdsapirc).
c = cdsapi.Client(url="https://ewds.climate.copernicus.eu/api")

year = "2020"
c.retrieve(
    "cems-glofas-historical",
    {
        "system_version": ["version_4_0"],
        "hydrological_model": ["lisflood"],
        "product_type": ["consolidated"],
        "variable": ["river_discharge_in_the_last_24_hours"],
        "hyear": [year],
        "hmonth": [f"{m:02d}" for m in range(1, 13)],
        "hday": [f"{d:02d}" for d in range(1, 32)],
        "data_format": "netcdf",
        "download_format": "zip",
    },
).download(f"glofas_v4_discharge_{year}.zip")


Repeat for each year you need. After unzipping, rename or organize files so the preprocessing notebook can find yearly NetCDFs named like `glofas_v4_discharge_YYYY.nc` with variable `dis24` and CF-compliant time/lat/lon coordinates.

For large downloads or regional subsets, see the [EWDS API examples](https://confluence.ecmwf.int/display/CEMS/EWDS+API) and [subsetting guidance](https://confluence.ecmwf.int/display/CEMS/Extract+subset+of+CEMS-Flood+Data).

### 3. Download LDD and upstream-area auxiliary data

The preprocessing step needs GloFAS **Large-scale Drainage Direction (LDD)** and **upstream area** fields to place river mouths on coastal cells. These auxiliary datasets are documented under [CEMS Auxiliary Data](https://confluence.ecmwf.int/spaces/CEMS/pages/242067380/Auxiliary+Data). Place them alongside the yearly discharge files, for example:

```text
glofas_data/
├── glofas_v4_discharge_2019.nc
├── glofas_v4_discharge_2020.nc
├── ...
├── ldd_glofas_v4_0.nc
└── uparea_glofas_v4_0.nc
```

### 4. Preprocess for ROMS-Tools

Raw GloFAS grids are **not** read directly by `RiverForcing`. Convert them to a Dai-compatible station NetCDF (`lat_mou`, `lon_mou`, `FLOW`, `ratio_m2s`, `riv_name`, optional `vol_stn`) using the [GloFAS preprocessing notebook](process_GloFAS.ipynb). Then pass the output file as:

```python
source = {
    "name": "GLOFAS",
    "path": "/path/to/glofas_v4_rivers_daily.nc",
}
```

See also the [river forcing notebook](river_forcing.ipynb) for usage examples.


## Downloading RIVR2O data

RIVR2O provides **annual biogeochemical river export** fields used as the dynamic BGC source for river forcing (`include_bgc=True`, `bgc_source={"name": "RIVR2O", "path": ...}`). Remaining MARBL tracers are filled from the recommended constants in `river_tracer_defaults.nc`.

### 1. Download from Zenodo

Download the RivR2O product from Zenodo:

- Dataset: [Biogeochemical river inputs for global ocean models (RivR2O)](https://zenodo.org/records/14889524)
- Canonical DOI: [10.5281/zenodo.13799103](https://doi.org/10.5281/zenodo.13799103)

Prefer the latest corrected release that includes the historical annual NetCDF files. After download, keep **one file per year**.

### 2. Required file naming and variables

`ROMS-Tools` parses the calendar year from the filename. Files must match:

```text
rivr2o_riverinputs_YYYY.nc
```

for example `rivr2o_riverinputs_2000.nc`. Wildcards and file lists are both supported:

```python
bgc_source = {
    "name": "RIVR2O",
    "path": "/path/to/rivr2o_riverinputs_*.nc",  # or a list of paths
}
```

Each yearly file should contain:

| Variable | Description |
|----------|-------------|
| `lat`, `lon` | Regular lat/lon grid |
| `DIC`, `DOC_l`, `DOC_sl`, `POC` | Carbon export (10⁶ g C yr⁻¹) |
| `DIN`, `DIP` | Nitrogen and phosphorus export (mapped internally to `NO3` and `PO4`) |

### 3. Coverage and usage notes

- Supported years in `ROMS-Tools` are **1903–2024**. Requests outside that range use the boundary-year values.
- Interior missing years are linearly interpolated along the calendar-year axis.
- Concentrations are derived at each river's coastal injection point from the nearest cell with positive DIC export; co-located rivers share export in proportion to discharge.
- For worked examples (including GloFAS discharge + RIVR2O BGC), see the [river forcing notebook](river_forcing.ipynb).
